# Домашня робота №3. Bank Marketing

Я досліджую результати телефонної маркетингової кампанії банку: як характеристики записів пов'язані з оформленням депозиту. Для цього я завантажую дані в PostgreSQL, очищую їх і виконую SQL-запити.

Я використовую повний `bank-full.csv`: 45 211 записів, 16 ознак і цільова колонка `y`. Я не роблю випадкової вибірки й не навчаю модель.

Джерело: [Bank Marketing, UCI](https://archive.ics.uci.edu/dataset/222/bank+marketing). Автори: S. Moro, P. Rita, P. Cortez, 2014. DOI: [10.24432/C5K306](https://doi.org/10.24432/C5K306). Ліцензія: CC BY 4.0.

Для запуску в Colab я обираю стандартне середовище CPU та виконую всі клітинки по черзі. Дані й залежності завантажую кодом, без підключення Google Drive.

## 1. Підготовка PostgreSQL

Спочатку я пробую запустити `pgserver`. Якщо для версії Python немає готового пакета, у Colab я встановлюю PostgreSQL через `apt`. У результаті я показую спосіб запуску та версію сервера.


In [1]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Я встановлюю лише відсутні залежності в поточне ядро.
packages = {"sqlalchemy": "sqlalchemy", "psycopg2": "psycopg2-binary", "pandas": "pandas"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if importlib.util.find_spec("pgserver") is None and sys.version_info[:2] <= (3, 12):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pgserver==0.1.4"], check=True)

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

if "engine" in globals():
    engine.dispose()

if importlib.util.find_spec("pgserver") is not None:
    import pgserver
    pg = pgserver.get_server("/tmp/hw3_bank_pg")
    DB_URI = pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1)
    server_method = "pgserver"
elif IN_COLAB and shutil.which("apt-get"):
    # Я створюю окремий тимчасовий кластер, не змінюючи пароль системного postgres.
    if not list(Path("/usr/lib/postgresql").glob("*/bin/initdb")):
        env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        subprocess.run(["apt-get", "-qq", "update"], check=True, env=env, stdout=subprocess.DEVNULL)
        subprocess.run(["apt-get", "-qq", "install", "-y", "postgresql"], check=True, env=env, stdout=subprocess.DEVNULL)
    pg_bin = sorted(Path("/usr/lib/postgresql").glob("*/bin/initdb"), key=lambda p: int(p.parents[1].name))[-1].parent
    pg_root = Path("/tmp/hw3_bank_apt")
    pg_data = pg_root / "data"
    pg_socket = pg_root / "socket"
    pg_socket.mkdir(parents=True, exist_ok=True)
    subprocess.run(["chown", "-R", "postgres:postgres", str(pg_root)], check=True)
    def as_postgres(*args):
        return subprocess.run(["runuser", "-u", "postgres", "--", *map(str, args)], check=True, capture_output=True, text=True)
    if not (pg_data / "PG_VERSION").exists():
        as_postgres(pg_bin / "initdb", "-D", pg_data, "--auth-local=trust", "--auth-host=reject", "--encoding=UTF8", "--locale=C.UTF-8")
    status = subprocess.run(["runuser", "-u", "postgres", "--", str(pg_bin / "pg_ctl"), "-D", str(pg_data), "status"], capture_output=True)
    if status.returncode:
        as_postgres(pg_bin / "pg_ctl", "-D", pg_data, "-l", pg_root / "server.log", "-o", f"-k {pg_socket} -p 55433 -c listen_addresses=''", "-w", "start")
    DB_URI = URL.create("postgresql+psycopg2", username="postgres", database="postgres", query={"host": str(pg_socket), "port": "55433"})
    server_method = "PostgreSQL через apt (Colab)"
else:
    raise RuntimeError("Я використовую Google Colab або локальне ядро Python 3.11/3.12 з pgserver.")

engine = create_engine(DB_URI, connect_args={"connect_timeout": 15})

def run(sql):
    with engine.begin() as connection:
        connection.execute(text(sql))

def query(sql):
    with engine.connect() as connection:
        return pd.read_sql(text(sql), connection)

def show(sql):
    result = query(sql)
    print(result.astype(object).where(result.notna(), "NULL").to_string(index=False))
    return result

print("Я запускаю сервер:", server_method)
print("Python:", sys.version.split()[0])
version_result = show("SELECT version() AS postgresql_version")


Я запускаю сервер: pgserver
Python: 3.11.16
                                                                                       postgresql_version
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 2. Отримання даних і COPY

Я завантажую офіційний архів UCI та знаходжу в ньому `bank-full.csv`. Я зберігаю початковий файл із роздільником `;`, перевіряю назви колонок і кількість записів. Я також рахую SHA-256, щоб можна було порівняти файл між запусками.


In [2]:
import csv
import hashlib
import io
import urllib.request
import zipfile

DATA_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
csv_path = DATA_DIR / "bank-full.csv"
source_columns = ["age", "job", "marital", "education", "default", "balance", "housing", "loan", "contact", "day", "month", "duration", "campaign", "pdays", "previous", "poutcome", "y"]

def find_csv(payload):
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for name in archive.namelist():
            if Path(name).name == "bank-full.csv":
                return archive.read(name)
        for name in archive.namelist():
            if Path(name).name == "bank.zip":
                return find_csv(archive.read(name))
    raise ValueError("Я не знайшов bank-full.csv в офіційному архіві.")

with urllib.request.urlopen(DATA_URL, timeout=90) as response:
    csv_bytes = find_csv(response.read())
csv_path.write_bytes(csv_bytes)
with csv_path.open(encoding="utf-8", newline="") as file:
    reader = csv.reader(file, delimiter=";")
    assert next(reader) == source_columns, "Змінилася структура джерела."
    file_rows = sum(1 for row in reader)
assert file_rows == 45_211, f"Змінилася кількість рядків: {file_rows}"
print("Я завантажив рядків:", file_rows)
print("Колонок:", len(source_columns))
print("Розмір CSV, байтів:", len(csv_bytes))
print("SHA-256:", hashlib.sha256(csv_bytes).hexdigest())


Я завантажив рядків: 45211
Колонок: 17
Розмір CSV, байтів: 4610348
SHA-256: d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d


In [3]:
# Я перестворюю лише дві власні таблиці, тому повторний запуск не додає дублікатів.
run("DROP TABLE IF EXISTS hw3_bank_clean; DROP TABLE IF EXISTS hw3_bank_staging;")
raw_columns = [name + "_raw" for name in source_columns]
run("CREATE TABLE hw3_bank_staging (" + ", ".join(name + " TEXT" for name in raw_columns) + ")")

raw_connection = engine.raw_connection()
try:
    with raw_connection.cursor() as cursor:
        with csv_path.open(encoding="utf-8", newline="") as file:
            cursor.copy_expert(
                "COPY hw3_bank_staging (" + ", ".join(raw_columns) + ") FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ';', ENCODING 'UTF8')",
                file,
            )
    raw_connection.commit()
except Exception:
    raw_connection.rollback()
    raise
finally:
    raw_connection.close()

loaded = show("SELECT COUNT(*) AS loaded_rows FROM hw3_bank_staging")
assert int(loaded.iloc[0, 0]) == file_rows


loaded_rows
      45211


Я звіряю кількість рядків у CSV та staging через `assert`. Так я перевіряю, що під час COPY не загубив жодного запису. На цьому кроці я ще не змінюю категорії та не привожу значення до числових типів.

## 3. Типи та правила очищення

Я використовую наведені нижче правила для старого варіанта `bank-full.csv`, а не для `bank-additional-full.csv`.

| Колонка | Що я зберігаю | Формат джерела | Тип і правило |
|---|---|---|---|
| `record_id` | Технічний номер рядка | Я створюю його під час імпорту | `BIGINT IDENTITY PRIMARY KEY`; я не вважаю його ідентифікатором клієнта |
| `age` | Вік у роках | Ціле число текстом | `SMALLINT NOT NULL`, я перевіряю `age > 0` |
| `job` | Професію | Категорія | `TEXT`, я перетворюю `unknown`, `?`, порожнє значення на `NULL` |
| `marital` | Сімейний стан | Категорія | `TEXT NOT NULL`, я прибираю пробіли; `divorced` включає також овдовілих |
| `education` | Рівень освіти | Категорія | `TEXT`, я нормалізую маркери пропусків |
| `has_default` | Наявність кредитного дефолту | `default`: `yes/no` | `BOOLEAN NOT NULL`, я перетворюю `yes/no` на `TRUE/FALSE` |
| `balance` | Середній річний баланс у євро | Ціле число текстом | `NUMERIC(12,2) NOT NULL`, я залишаю від'ємні баланси; точний тип придатний для грошей |
| `has_housing_loan` | Наявність житлового кредиту | `housing`: `yes/no` | `BOOLEAN NOT NULL`, я нормалізую `yes/no` |
| `has_personal_loan` | Наявність особистого кредиту | `loan`: `yes/no` | `BOOLEAN NOT NULL`, я нормалізую `yes/no` |
| `contact` | Канал останнього контакту | Категорія | `TEXT`, я нормалізую маркери пропусків |
| `contact_day` | День місяця останнього контакту | `day`: ціле число | `SMALLINT NOT NULL`, я перевіряю діапазон 1–31 |
| `contact_month` | Місяць останнього контакту | `month`: `jan`–`dec` | `TEXT NOT NULL`, я перевіряю перелік місяців |
| `duration_seconds` | Тривалість останнього дзвінка | `duration`: ціле число секунд | `INTEGER NOT NULL`, я перевіряю `>= 0` |
| `campaign` | Кількість контактів поточної кампанії, включно з останнім | Ціле число | `INTEGER NOT NULL`, я перевіряю `>= 1` |
| `pdays` | Дні після контакту попередньої кампанії | Ціле число, `-1` означає відсутність контакту | `INTEGER`, я перетворюю `-1` на `NULL`, для решти перевіряю `>= 0` |
| `previous` | Кількість контактів до поточної кампанії | Ціле число | `INTEGER NOT NULL`, я перевіряю `>= 0` |
| `poutcome` | Результат попередньої кампанії | Категорія | `TEXT`, я нормалізую маркери пропусків; `other` залишаю окремою категорією |
| `subscribed` | Чи оформив клієнт депозит | `y`: `yes/no` | `BOOLEAN NOT NULL`, я використовую як цільову ознаку |

Я застосовую `TRIM` до всіх сирих полів. Я не вигадую повну дату: у файлі немає року, тому `DATE` і `TIMESTAMPTZ` тут недоречні. Я не ставлю довільних обмежень довжини тексту. Я відрізняю `pdays IS NULL` через відсутність попереднього контакту від невідомої категорії `contact`.

Перед приведенням типів я перевіряю формат і допустимі діапазони. Якщо джерело зміниться, я зупиню імпорт із підрахунком проблем, а не відкидатиму записи мовчки.


In [4]:
# Я перевіряю формат, діапазон і обов'язковість полів до приведення типів.
validation_rules = {
    "age": "CASE WHEN TRIM(age_raw) ~ '^[0-9]{1,5}$' THEN TRIM(age_raw)::INTEGER BETWEEN 1 AND 32767 ELSE FALSE END",
    "balance": "CASE WHEN TRIM(balance_raw) ~ '^-?[0-9]{1,10}$' THEN ABS(TRIM(balance_raw)::NUMERIC) < 10000000000 ELSE FALSE END",
    "day": "CASE WHEN TRIM(day_raw) ~ '^[0-9]{1,2}$' THEN TRIM(day_raw)::INTEGER BETWEEN 1 AND 31 ELSE FALSE END",
    "duration": "CASE WHEN TRIM(duration_raw) ~ '^[0-9]{1,10}$' THEN TRIM(duration_raw)::NUMERIC BETWEEN 0 AND 2147483647 ELSE FALSE END",
    "campaign": "CASE WHEN TRIM(campaign_raw) ~ '^[0-9]{1,10}$' THEN TRIM(campaign_raw)::NUMERIC BETWEEN 1 AND 2147483647 ELSE FALSE END",
    "pdays": "CASE WHEN TRIM(pdays_raw) ~ '^-?[0-9]{1,10}$' THEN TRIM(pdays_raw)::NUMERIC BETWEEN -1 AND 2147483647 ELSE FALSE END",
    "previous": "CASE WHEN TRIM(previous_raw) ~ '^[0-9]{1,10}$' THEN TRIM(previous_raw)::NUMERIC BETWEEN 0 AND 2147483647 ELSE FALSE END",
    "marital": "NULLIF(TRIM(marital_raw), '') IS NOT NULL AND LOWER(TRIM(marital_raw)) NOT IN ('unknown', '?')",
    "month": "TRIM(month_raw) IN ('jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec')",
}
for name in ["default", "housing", "loan", "y"]:
    validation_rules[name] = f"LOWER(TRIM({name}_raw)) IN ('yes', 'no')"
validation_sql = " UNION ALL ".join(
    f"SELECT '{name}' AS column_name, COUNT(*) FILTER (WHERE ({rule}) IS NOT TRUE) AS invalid_rows FROM hw3_bank_staging"
    for name, rule in validation_rules.items()
)
invalid_values = show(validation_sql)
assert int(invalid_values["invalid_rows"].sum()) == 0, "Я зупиняю імпорт: у staging є некоректні обов'язкові значення."


column_name invalid_rows
          y            0
       loan            0
    housing            0
    default            0
      month            0
    marital            0
    balance            0
   duration            0
      pdays            0
        age            0
        day            0
   campaign            0
   previous            0


In [5]:
run("""
CREATE TABLE hw3_bank_clean (
    record_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    age SMALLINT NOT NULL CHECK (age > 0),
    job TEXT,
    marital TEXT NOT NULL,
    education TEXT,
    has_default BOOLEAN NOT NULL,
    balance NUMERIC(12,2) NOT NULL,
    has_housing_loan BOOLEAN NOT NULL,
    has_personal_loan BOOLEAN NOT NULL,
    contact TEXT,
    contact_day SMALLINT NOT NULL CHECK (contact_day BETWEEN 1 AND 31),
    contact_month TEXT NOT NULL CHECK (contact_month IN ('jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec')),
    duration_seconds INTEGER NOT NULL CHECK (duration_seconds >= 0),
    campaign INTEGER NOT NULL CHECK (campaign >= 1),
    pdays INTEGER CHECK (pdays IS NULL OR pdays >= 0),
    previous INTEGER NOT NULL CHECK (previous >= 0),
    poutcome TEXT,
    subscribed BOOLEAN NOT NULL
);

INSERT INTO hw3_bank_clean (
    age, job, marital, education, has_default, balance,
    has_housing_loan, has_personal_loan, contact, contact_day,
    contact_month, duration_seconds, campaign, pdays, previous, poutcome, subscribed
)
SELECT
    TRIM(age_raw)::SMALLINT,
    CASE WHEN LOWER(TRIM(job_raw)) IN ('', '?', 'unknown') THEN NULL ELSE TRIM(job_raw) END,
    TRIM(marital_raw),
    CASE WHEN LOWER(TRIM(education_raw)) IN ('', '?', 'unknown') THEN NULL ELSE TRIM(education_raw) END,
    LOWER(TRIM(default_raw)) = 'yes',
    TRIM(balance_raw)::NUMERIC(12,2),
    LOWER(TRIM(housing_raw)) = 'yes',
    LOWER(TRIM(loan_raw)) = 'yes',
    CASE WHEN LOWER(TRIM(contact_raw)) IN ('', '?', 'unknown') THEN NULL ELSE TRIM(contact_raw) END,
    TRIM(day_raw)::SMALLINT,
    TRIM(month_raw),
    TRIM(duration_raw)::INTEGER,
    TRIM(campaign_raw)::INTEGER,
    NULLIF(TRIM(pdays_raw)::INTEGER, -1),
    TRIM(previous_raw)::INTEGER,
    CASE WHEN LOWER(TRIM(poutcome_raw)) IN ('', '?', 'unknown') THEN NULL ELSE TRIM(poutcome_raw) END,
    LOWER(TRIM(y_raw)) = 'yes'
FROM hw3_bank_staging;
""")
preview = show("SELECT record_id, age, job, balance, contact, pdays, subscribed FROM hw3_bank_clean ORDER BY record_id LIMIT 5")


record_id age          job balance contact pdays subscribed
        1  58   management  2143.0    NULL  NULL      False
        2  44   technician    29.0    NULL  NULL      False
        3  33 entrepreneur     2.0    NULL  NULL      False
        4  47  blue-collar  1506.0    NULL  NULL      False
        5  33         NULL     1.0    NULL  NULL      False


## 4. Аудит якості

### 4.1. Кількість рядків

Я порівнюю сирий і очищений рівні. Я не застосовую фільтрів видалення: якщо перевірка формату пройдена, кожен сирий рядок має потрапити у фінальну таблицю.


In [6]:
row_counts = show("""
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_bank_staging
UNION ALL
SELECT 'clean' AS layer, COUNT(*) AS n_rows FROM hw3_bank_clean;
""")
assert row_counts["n_rows"].tolist() == [file_rows, file_rows]


  layer n_rows
staging  45211
  clean  45211


Я отримав по 45 211 рядків у staging і clean. Я не відфільтрував жодного запису: усі значення пройшли перевірку формату й діапазонів. Я підтвердив це також порівнянням із кількістю рядків у CSV.


### 4.2. Дублікати

Я не маю номера клієнта або гарантованого природного ключа. Тому я використовую комбінацію всіх 17 вихідних атрибутів як наближення до business key. Я шукаю лише точні дублікати записів, а не повторних клієнтів; технічний `record_id` у перевірку не включаю.


In [7]:
duplicates = show("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (age, job, marital, education, has_default, balance,
        has_housing_loan, has_personal_loan, contact, contact_day, contact_month,
        duration_seconds, campaign, pdays, previous, poutcome, subscribed)) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (age, job, marital, education, has_default, balance,
        has_housing_loan, has_personal_loan, contact, contact_day, contact_month,
        duration_seconds, campaign, pdays, previous, poutcome, subscribed)) AS duplicate_rows
FROM hw3_bank_clean;
""")


total_rows distinct_business_keys duplicate_rows
     45211                  45211              0


Я отримав 45 211 різних комбінацій вихідних атрибутів і нуль точних дублікатів. Я зберіг усі рядки, але не роблю висновку про унікальність клієнтів: їхніх ідентифікаторів у файлі немає.


### 4.3. Сирі маркери й NULL

Я перевіряю `unknown`, `?`, порожній рядок і справжній SQL `NULL` в усіх сирих колонках. Окремо я рахую `pdays = -1`, бо це змістовний маркер відсутності попереднього контакту.


In [8]:
sentinel_sql = " UNION ALL ".join(
    f"SELECT '{name}' AS column_name, COUNT(*) FILTER (WHERE {name}_raw IS NULL) AS sql_null, "
    f"COUNT(*) FILTER (WHERE TRIM({name}_raw) = '') AS empty_value, "
    f"COUNT(*) FILTER (WHERE TRIM({name}_raw) = '?') AS question_mark, "
    f"COUNT(*) FILTER (WHERE LOWER(TRIM({name}_raw)) = 'unknown') AS unknown_value "
    "FROM hw3_bank_staging"
    for name in source_columns
)
sentinels = show(sentinel_sql)
pdays_sentinel = show("SELECT COUNT(*) FILTER (WHERE TRIM(pdays_raw) = '-1') AS no_previous_contact FROM hw3_bank_staging")
null_rates = show("""
SELECT v.column_name, COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE v.is_missing) AS null_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE v.is_missing) / NULLIF(COUNT(*), 0), 2) AS pct_null
FROM hw3_bank_clean AS b
CROSS JOIN LATERAL (VALUES
    ('job', b.job IS NULL), ('education', b.education IS NULL),
    ('contact', b.contact IS NULL), ('poutcome', b.poutcome IS NULL),
    ('pdays', b.pdays IS NULL), ('age', b.age IS NULL)
) AS v(column_name, is_missing)
GROUP BY v.column_name
ORDER BY pct_null DESC, v.column_name;
""")


column_name sql_null empty_value question_mark unknown_value
        age        0           0             0             0
          y        0           0             0             0
   poutcome        0           0             0         36959
   previous        0           0             0             0
      pdays        0           0             0             0
   campaign        0           0             0             0
    marital        0           0             0             0
  education        0           0             0          1857
    balance        0           0             0             0
       loan        0           0             0             0
        day        0           0             0             0
      month        0           0             0             0
        job        0           0             0           288
    default        0           0             0             0
    housing        0           0             0             0
    contact        0    

column_name total_rows null_rows pct_null
   poutcome      45211     36959    81.75
      pdays      45211     36954    81.74
    contact      45211     13020     28.8
  education      45211      1857     4.11
        job      45211       288     0.64
        age      45211         0      0.0


Я знайшов `unknown` у 36 959 значеннях `poutcome`, 13 020 значеннях `contact`, 1 857 значеннях `education` і 288 значеннях `job`. Порожніх рядків, `?` і сирих SQL `NULL` я не виявив. Я отримав 81,74% `NULL` у `pdays` через 36 954 маркери `-1`; це відсутність попереднього контакту. У `age` я отримав нульовий null-rate, але не вважаю це гарантією повної якості.


## 5. Розподіли й потенційні аномалії

### 5.1. Розподіл професій

Я рахую десять найпоширеніших категорій, включно з невідомими значеннями. Відсоток я обчислюю від усієї таблиці до застосування `LIMIT`.


In [9]:
job_distribution = show("""
SELECT COALESCE(job, '(невідомо)') AS job, COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_bank_clean
GROUP BY job
ORDER BY n_rows DESC, job
LIMIT 10;
""")


          job n_rows pct_rows
  blue-collar   9732    21.53
   management   9458    20.92
   technician   7597     16.8
       admin.   5171    11.44
     services   4154     9.19
      retired   2264     5.01
self-employed   1579     3.49
 entrepreneur   1487     3.29
   unemployed   1303     2.88
    housemaid   1240     2.74


Я бачу найбільші групи `blue-collar` — 9 732 записи (21,53%) та `management` — 9 458 (20,92%). Разом вони становлять 42,45% набору. Я враховую цю нерівномірність: загальний результат сильніше залежить від великих груп, ніж від рідкісних професій.


### 5.2. Статистика балансу

Я перевіряю середній річний баланс у євро. Від'ємні суми я не вважаю автоматично помилкою: баланс може бути від'ємним.


In [10]:
balance_stats = show("""
SELECT COUNT(balance) AS n_non_null, MIN(balance) AS min_value,
    MAX(balance) AS max_value, ROUND(AVG(balance), 2) AS avg_value,
    ROUND(STDDEV_SAMP(balance), 2) AS sd_value,
    ROUND((PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY balance))::NUMERIC, 2) AS median_value
FROM hw3_bank_clean;
""")


n_non_null min_value max_value avg_value sd_value median_value
     45211   -8019.0  102127.0   1362.27  3044.77        448.0


Я отримав баланс від −8 019 до 102 127 євро. Середнє становить 1 362,27 євро, медіана — 448, а стандартне відхилення — 3 044,77. За цією різницею я бачу довгий правий хвіст і не покладаюся лише на середнє.


### 5.3. Перевірка за IQR

Я шукаю баланс нижче `Q1 − 1.5 × IQR` або вище `Q3 + 1.5 × IQR`, де `IQR = Q3 − Q1`. Так я позначаю потенційні аномалії, а не доведені помилки. Я використовую IQR, бо великі баланси можуть сильно зміщувати середнє й стандартне відхилення.


In [11]:
iqr_summary = show("""
WITH quartiles AS (
    SELECT PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY balance) AS q1,
           PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY balance) AS q3
    FROM hw3_bank_clean
)
SELECT q1, q3, q3 - q1 AS iqr,
    q1 - 1.5 * (q3 - q1) AS lower_bound,
    q3 + 1.5 * (q3 - q1) AS upper_bound,
    COUNT(*) FILTER (WHERE balance < q1 - 1.5 * (q3 - q1)
                       OR balance > q3 + 1.5 * (q3 - q1)) AS flagged_rows
FROM hw3_bank_clean CROSS JOIN quartiles
GROUP BY q1, q3;
""")
iqr_examples = show("""
WITH quartiles AS (
    SELECT PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY balance) AS q1,
           PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY balance) AS q3
    FROM hw3_bank_clean
)
SELECT b.record_id, b.age, b.job, b.balance, b.subscribed
FROM hw3_bank_clean AS b CROSS JOIN quartiles AS q
WHERE b.balance < q.q1 - 1.5 * (q.q3 - q.q1)
   OR b.balance > q.q3 + 1.5 * (q.q3 - q.q1)
ORDER BY b.balance DESC, b.record_id
LIMIT 20;
""")


   q1      q3     iqr lower_bound upper_bound flagged_rows
 72.0  1428.0  1356.0     -1962.0      3462.0         4729
record_id age           job   balance subscribed
    39990  51    management  102127.0      False
    26228  59    management   98417.0      False
    42559  84       retired   81204.0       True
    43394  84       retired   81204.0       True
    41694  60       retired   71188.0      False
    19786  56    management   66721.0      False
    21193  52   blue-collar   66653.0      False
    19421  59        admin.   64343.0      False
    41375  32  entrepreneur   59649.0      False
    12927  56   blue-collar   58932.0      False
      921  44        admin.   58544.0      False
    27391  50      services   57435.0      False
     3198  43        admin.   56831.0      False
    40764  61 self-employed   52587.0       True
    43054  61 self-employed   52587.0       True
    22092  40    management   52527.0      False
     9537  58  entrepreneur   52499.0      False


Я отримав Q1 = 72, Q3 = 1 428 та IQR = 1 356 євро. За межами від −1 962 до 3 462 євро я знайшов 4 729 записів, тобто 10,46% набору. Я залишаю їх у таблиці й запропонував би додати прапорець для перевірки джерела: великий або від’ємний баланс сам по собі не доводить помилку.


## 6. Десять DQL / EDA-запитів

Я відокремлюю ці десять запитів від аудиту. У висновках я описую зв'язки в цьому файлі, а не причинний вплив ознак.


### Запит 1

**Моє бізнес-питання:** Яку частку підписок я отримав у наборі?


In [12]:
q1 = show("""
SELECT subscribed, COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_bank_clean
GROUP BY subscribed
ORDER BY subscribed DESC;
""")


subscribed n_rows pct_rows
      True   5289     11.7
     False  39922     88.3


**Мій висновок:** Я отримав 5 289 підписок (11,70%) проти 39 922 відмов (88,30%). Я бачу дисбаланс цільових класів, тому для майбутньої моделі не обмежився б accuracy: постійний прогноз відмови вже дав би 88,30%.


### Запит 2

**Моє бізнес-питання:** Для яких професій я бачу найбільшу частку підписок серед груп від 100 записів?


In [13]:
q2 = show("""
SELECT job, COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE subscribed) AS subscriptions,
    ROUND(100.0 * COUNT(*) FILTER (WHERE subscribed) / NULLIF(COUNT(*), 0), 2) AS subscription_pct
FROM hw3_bank_clean
WHERE job IS NOT NULL
GROUP BY job
HAVING COUNT(*) >= 100
ORDER BY subscription_pct DESC, n_rows DESC, job
LIMIT 5;
""")


       job n_rows subscriptions subscription_pct
   student    938           269            28.68
   retired   2264           516            22.79
unemployed   1303           202             15.5
management   9458          1301            13.76
    admin.   5171           631             12.2


**Мій висновок:** Найвищу частку підписок я бачу у `student` — 28,68% із 938 записів і `retired` — 22,79% із 2 264. Для `management` я отримав нижчу частку 13,76%, але більше підписок за кількістю — 1 301. Я розрізняю частку успіхів і їхню абсолютну кількість та не трактую професію як причину результату.


### Запит 3

**Моє бізнес-питання:** Яку різницю в частці підписок я бачу між каналами контакту?


In [14]:
q3 = show("""
SELECT COALESCE(contact, '(невідомо)') AS contact, COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE subscribed) / NULLIF(COUNT(*), 0), 2) AS subscription_pct
FROM hw3_bank_clean
GROUP BY contact
ORDER BY n_rows DESC, contact;
""")


   contact n_rows subscription_pct
  cellular  29285            14.92
(невідомо)  13020             4.07
 telephone   2906            13.42


**Мій висновок:** Для `cellular` я отримав 14,92% підписок із 29 285 записів, для `telephone` — 13,42% із 2 906. У 13 020 записах із невідомим каналом я бачу лише 4,07%. Я зберіг би ознаку відсутнього каналу для подальшого аналізу, але перевірив би, чи різницю не пояснює період кампанії.


### Запит 4

**Моє бізнес-питання:** Яку різницю в частці підписок я бачу між віковими групами?


In [15]:
q4 = show("""
SELECT CASE WHEN age BETWEEN 18 AND 29 THEN '18–29'
            WHEN age BETWEEN 30 AND 44 THEN '30–44'
            WHEN age BETWEEN 45 AND 59 THEN '45–59'
            WHEN age >= 60 THEN '60+'
            ELSE 'до 18' END AS age_group,
    COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE subscribed) / NULLIF(COUNT(*), 0), 2) AS subscription_pct
FROM hw3_bank_clean
GROUP BY age_group
ORDER BY subscription_pct DESC, age_group;
""")


age_group n_rows subscription_pct
      60+   1784            33.63
    18–29   5273             17.6
    30–44  24274            10.12
    45–59  13880              9.4


**Мій висновок:** У групі 60+ я отримав найвищу частку підписок — 33,63%, але лише 1 784 записи. Найбільша група 30–44 містить 24 274 записи й має 10,12% підписок. Я порівнюю частки разом з обсягами груп і не переношу цей зв’язок автоматично на всіх клієнтів банку.


### Запит 5

**Моє бізнес-питання:** Який зв’язок я бачу між часткою підписок і кількістю контактів поточної кампанії?


In [16]:
q5 = show("""
SELECT CASE WHEN campaign = 1 THEN '1'
            WHEN campaign BETWEEN 2 AND 3 THEN '2–3'
            WHEN campaign BETWEEN 4 AND 5 THEN '4–5'
            ELSE '6+' END AS campaign_group,
    COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE subscribed) / NULLIF(COUNT(*), 0), 2) AS subscription_pct
FROM hw3_bank_clean
GROUP BY campaign_group
ORDER BY subscription_pct DESC, campaign_group;
""")


campaign_group n_rows subscription_pct
             1  17544             14.6
           2–3  18026             11.2
           4–5   5286             8.63
            6+   4355             5.81


**Мій висновок:** Для одного контакту я отримав 14,60% підписок, а для шести й більше — 5,81%. Для проміжних груп я бачу 11,20% і 8,63%. Я не роблю висновку, що повторні дзвінки спричиняють відмову: додаткові контакти могли бути потрібні саме складнішим випадкам.


### Запит 6

**Моє бізнес-питання:** Які найбільші баланси я бачу серед людей віком 30–50 років без дефолту та без хоча б одного з двох кредитів?


In [17]:
q6 = show("""
SELECT record_id, age, job, balance, has_housing_loan, has_personal_loan, subscribed
FROM hw3_bank_clean
WHERE age BETWEEN 30 AND 50
  AND NOT has_default
  AND (NOT has_housing_loan OR NOT has_personal_loan)
ORDER BY balance DESC, record_id
LIMIT 10;
""")


record_id age          job  balance has_housing_loan has_personal_loan subscribed
    41375  32 entrepreneur  59649.0            False             False      False
      921  44       admin.  58544.0             True             False      False
    27391  50     services  57435.0             True             False      False
     3198  43       admin.  56831.0            False             False      False
    22092  40   management  52527.0            False             False      False
    15870  38       admin.  45789.0            False             False      False
      447  39   technician  45248.0             True             False       True
    27050  31   management  44128.0            False             False      False
    19796  42 entrepreneur  42045.0            False             False      False
    21347  43   management  41630.0            False             False      False


**Мій висновок:** Серед відібраних записів я бачу найбільший баланс 59 649 євро у 32-річного клієнта з категорії `entrepreneur`. У десяти показаних рядках я знайшов одну підписку — за балансу 45 248 євро. Я читаю умову в дужках як відсутність хоча б одного кредиту; результат не означає відсутності обох кредитів і не описує всю вибірку.


### Запит 7

**Моє бізнес-питання:** Які різні назви професій містять фрагмент admin і скільки записів я для них бачу?


In [18]:
q7 = show("""
SELECT DISTINCT job,
    COUNT(*) OVER (PARTITION BY job) AS n_rows
FROM hw3_bank_clean
WHERE job ILIKE '%admin%'
ORDER BY job;
""")


   job n_rows
admin.   5171


**Мій висновок:** Я знайшов одну різну назву професії — `admin.`, яка трапляється у 5 171 записі. Я використав `ILIKE` для пошуку без урахування регістру, а `DISTINCT` — для одного рядка на назву. Кількість перед усуненням повторів я обчислив віконною функцією `COUNT(*) OVER (PARTITION BY job)`.


### Запит 8

**Моє бізнес-питання:** Яку різницю в частці підписок я бачу за наявністю даних про освіту?


In [19]:
q8 = show("""
SELECT 'освіта невідома' AS education_status, COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE subscribed) / NULLIF(COUNT(*), 0), 2) AS subscription_pct
FROM hw3_bank_clean
WHERE education IS NULL
UNION ALL
SELECT 'освіта відома' AS education_status, COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE subscribed) / NULLIF(COUNT(*), 0), 2) AS subscription_pct
FROM hw3_bank_clean
WHERE education IS NOT NULL
ORDER BY education_status;
""")


education_status n_rows subscription_pct
   освіта відома  43354            11.62
 освіта невідома   1857            13.57


**Мій висновок:** Для 1 857 записів із невідомою освітою я отримав 13,57% підписок, для 43 354 записів із відомою — 11,62%. Я не відкидав би першу групу автоматично: пропуск теж може бути пов’язаний з іншими характеристиками записів.


### Запит 9

**Моє бізнес-питання:** Яку різницю в частці підписок я бачу між записами з попереднім успіхом і рештою?


In [20]:
q9 = show("""
SELECT poutcome IS NOT DISTINCT FROM 'success' AS previous_success,
    COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE subscribed) / NULLIF(COUNT(*), 0), 2) AS subscription_pct
FROM hw3_bank_clean
GROUP BY previous_success
ORDER BY previous_success DESC;
""")


previous_success n_rows subscription_pct
            True   1511            64.73
           False  43700             9.86


**Мій висновок:** Серед 1 511 записів із попереднім успіхом я отримав 64,73% підписок, серед решти 43 700 — 9,86%. Через `IS NOT DISTINCT FROM` я включив `NULL` у групу без підтвердженого успіху. Я не називаю всі ці випадки попередніми невдачами: частина взагалі не мала попереднього контакту.


### Запит 10

**Моє бізнес-питання:** Які найбільші баланси я бачу серед підписників із професій management і technician?


In [21]:
q10 = show("""
SELECT record_id, age, job, balance, campaign, subscribed
FROM hw3_bank_clean
WHERE subscribed
  AND job IN ('management', 'technician')
ORDER BY balance DESC, record_id
LIMIT 10;
""")


record_id age        job  balance campaign subscribed
      447  39 technician  45248.0        1       True
    34171  29 management  36252.0        1       True
    20738  32 technician  34646.0        9       True
    44233  45 management  31472.0        7       True
    40879  41 management  27696.0        2       True
    43020  41 management  27696.0        2       True
    43673  42 management  27696.0        2       True
    44522  42 management  27696.0        1       True
    39583  34 technician  23076.0        2       True
    42196  33 management  22867.0        2       True


**Мій висновок:** Найбільший баланс у цій десятці я бачу в `technician` — 45 248 євро; наступний у `management` — 36 252 євро. Я також бачу повторення балансу 27 696 євро в різних записах, але не вважаю їх дублями лише за цією сумою. Через фільтр підписок і `LIMIT 10` я не оцінюю за цією таблицею типового клієнта.


### Які конструкції я використав

| Вимога | Де я її показав |
|---|---|
| Явний перелік колонок | У всіх десяти запитах |
| `WHERE`, `AND`, `OR`, `NOT`, дужки | Запит 6 |
| `IN` / `BETWEEN` | Запити 4, 5, 6, 10 |
| `LIKE` / `ILIKE` | Запит 7 |
| `IS NULL` / `IS NOT NULL` | Запити 2, 8 |
| `COALESCE` / `NULLIF` | Запити 2–5, 8, 9 |
| `IS DISTINCT FROM` / `IS NOT DISTINCT FROM` | Запит 9 |
| `DISTINCT` | Запит 7 |
| `ORDER BY` разом із `LIMIT` | Запити 2, 6, 10 |
| Щонайменше два `GROUP BY` | Запити 1–5, 9 |

## 7. Reflection

Я завантажив повний файл Bank Marketing і зберіг усі записи після очищення. Я не знайшов точних дублікатів за комбінацією всіх вихідних полів, але не можу перевірити повторних клієнтів без їхнього ідентифікатора. Я виявив категорії unknown у професії, освіті, каналі контакту та результаті попередньої кампанії. Я перетворив їх на NULL. Для pdays я окремо врахував значення −1: воно означає відсутність попереднього контакту, а не випадково загублену кількість днів.

Я залишив від’ємні баланси й великі суми. За правилом IQR я позначив потенційні аномалії, але не отримав доказу помилки в цих записах. Перед навчанням я перевірив би їх разом із власником джерела. Я також врахував би дисбаланс підписок, щоб висока accuracy не приховала слабке виявлення позитивного класу.

Я розглядаю цю задачу як бінарну класифікацію: чи оформить клієнт депозит. Для прогнозу до початку кампанії я використав би доступні на той момент дані про вік, професію, освіту, кредити, баланс і попередні кампанії. Я не використав би технічний record_id та ціль subscribed як ознаки. Я виключив би duration_seconds, бо тривалість дзвінка відома лише після його завершення. Я також перевірив би момент доступності campaign, каналу й дня останнього контакту: у фінальному файлі вони можуть описувати майбутнє відносно прогнозу.

Я запропонував би data engineer додати ідентифікатори клієнта й контакту, повну дату, час формування ознак і версію схеми. Я також попросив би розділити невідоме значення та відсутність попередньої події, автоматично перевіряти діапазони й звіряти кількість рядків після кожного завантаження.

## 8. Підсумкова перевірка

Я повторно перевіряю кількість рядків і наявність усіх десяти результатів. Під час повторного запуску я перестворюю власні таблиці, тому імпорт не накопичує дані.


In [22]:
assert int(query("SELECT COUNT(*) AS n FROM hw3_bank_staging").iloc[0, 0]) == file_rows
assert int(query("SELECT COUNT(*) AS n FROM hw3_bank_clean").iloc[0, 0]) == file_rows
assert all(isinstance(globals()[f"q{i}"], pd.DataFrame) for i in range(1, 11))
print("Я виконав 10 EDA-запитів і перевірив кількість рядків у двох таблицях.")
print("Я завершив повний запуск без помилок.")


Я виконав 10 EDA-запитів і перевірив кількість рядків у двох таблицях.
Я завершив повний запуск без помилок.
